# Module 8 Assignment: Give Your Persona Agent Memory With Sessions

Lab 7 added a `SQLiteSession` to Study Buddy: first in memory, to fix two `Runner.run` calls that
did not remember each other, then file backed, so the conversation survives a kernel restart. The
Gradio `chat()` function got simpler in the process, no more rebuilding `history` by hand.

This assignment makes you do the same thing to your **own** Persona Agent, the one you rebuilt on
the OpenAI Agents SDK in the Module 7 assignment. Nothing about your agent's persona, notes, or
three tools needs to change here: you are giving it memory, not redesigning it.

**What you'll build:**
1. Your Module 7 assignment's Agent, Azure configuration, and three tools, carried over unchanged
2. A quick demonstration that two separate `Runner.run` calls do not remember each other
3. A file backed `SQLiteSession` that fixes it
4. A simplified Gradio `chat()` function that uses `session=session` instead of rebuilding `history`

**Provider:** Azure OpenAI via APIM, deployment `gpt-5.4-ptu`, same as every lab and assignment so
far.

If you get stuck on any step, look at how Lab 7 made the same kind of change to Study Buddy, and do
the analogous thing to your own agent.

## 0. Install the packages (run once)

Same as Lab 7: no new package beyond what your Module 7 assignment already installed.
`SQLiteSession` ships with `openai-agents` itself.

In [1]:
!uv pip install --upgrade openai openai-agents pypdf python-dotenv requests gradio

Resolved 79 packages in 950ms
Checked 79 packages in 8ms


## 1. Imports

**TODO:** Copy the imports from your Module 7 assignment, plus one new name: `SQLiteSession`,
imported from `agents` alongside `Agent` and `Runner`, the same way Lab 7, Step 1 added it.

In [2]:
# TODO: copy your imports from the Module 7 assignment, plus SQLiteSession from agents
from dotenv import load_dotenv
import os
import asyncio
import html
from typing import Literal, Optional

import requests
from openai import AsyncAzureOpenAI
from agents import (
    Agent,
    Runner,
    function_tool,
    OpenAIChatCompletionsModel,
    set_tracing_disabled,
    SQLiteSession,
)
from pypdf import PdfReader
import gradio as gr

c:\Users\estev\Assignment 8\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Load your Azure OpenAI credentials from `.env`

Identical to the Module 7 assignment. Sessions do not change which provider you talk to.

In [3]:
from dotenv import load_dotenv
import os

load_dotenv(override=True)

AZURE_OPENAI_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT")
AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY")
DEPLOYMENT = "gpt-5.4-ptu"

print("Endpoint loaded:", bool(AZURE_OPENAI_ENDPOINT))
print("API key loaded:", bool(AZURE_OPENAI_API_KEY))

Endpoint loaded: True
API key loaded: True


## 3. Configure the OpenAI Agents SDK for Azure OpenAI

**TODO:** Copy this setup from your Module 7 assignment, unchanged: `set_tracing_disabled(True)`,
your `AsyncAzureOpenAI` client, and `OpenAIChatCompletionsModel`. A session is a feature of
`Runner.run`, not of the model or the client, so nothing here needs to change.

In [10]:
# TODO: set_tracing_disabled(True), then build azure_client and azure_model, exactly like your Module 7 assignment
set_tracing_disabled(True)

azure_client = AsyncAzureOpenAI(
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_key=AZURE_OPENAI_API_KEY,
    api_version="2024-10-21",
)

azure_model = OpenAIChatCompletionsModel(
    model=DEPLOYMENT,
    openai_client=azure_client,
)

print("Azure model configured for the OpenAI Agents SDK.")


Azure model configured for the OpenAI Agents SDK.


## 4. Your persona, notes, and three tools

**TODO:** Bring over your persona, your notes, and all three of your `@function_tool` decorated
tools from the Module 7 assignment, unchanged. None of them need to change for this assignment: a
session only affects how conversation history is stored and replayed, not what a tool does.

In [11]:
# TODO: persona = "..." (a parenthesized multi-line string), from your Module 7 assignment
NOTES_PATH = "notes.pdf"

persona = """You're a professional, ethical, and fancy-worded Quiz Game Intructor."""

def load_notes(path):
    try:
        reader = PdfReader(path)
        text = "\n".join(page.extract_text() or "" for page in reader.pages)
        return text.strip()
    except FileNotFoundError:
        return "(notes.pdf not found yet: add it to this folder to enable real study content.)"

study_notes = load_notes(NOTES_PATH)
print("Persona loaded:", persona)
print(study_notes[:400])


Persona loaded: You're a professional, ethical, and fancy-worded Quiz Game Intructor.
types
 
of
 
video
 
games
 
genre:
 
 
-
 
Action
 
-
 
Adventure
 
-
 
RPG
 
-
 
Strategy
 
-
 
Simulation
 
-
 
Sports
 
-
 
Racing
 
-
 
Horror
 
-
 
Puzzle
 
-
 
Fighting
 
-
 
Platformer
 
-
 
Shooter
 
-
 
Survival
 
-
 
Stealth
 
-
 
Sandbox
 
-
 
Rhythm


In [12]:
# TODO: your first tool, decorated with @function_tool, from your Module 7 assignment

LOG_PATH = "missed_questions.txt"

@function_tool
def my_tool(value: str) -> str:
    """Log a missed question so the student can review it later.

    Args:
        value: The question or text to save for review.
    """
    print(f"Tool called to log a missed question: {value}")

    with open(LOG_PATH, "a", encoding="utf-8") as f:
        f.write(value + "\n")

    return "the missed questions is for review"

print("Tool ready:", my_tool.name)

Tool ready: my_tool


In [13]:
# TODO: your second tool, decorated with @function_tool, from your Module 7 assignment
@function_tool
def api_tool(value: str) -> str:
    """Fetch a bonus trivia question from a free external API.

    Args:
        value: The requested trivia category or topic.
    """
    print(f"Tool called to fetch a bonus trivia question: {value}")

    params = {"amount": 1, "type": "multiple", "category": 15}

    try:
        response = requests.get(
            "https://opentdb.com/api.php",
            params=params,
            timeout=20
        )
        response.raise_for_status()
        data = response.json()

    except (requests.exceptions.RequestException, ValueError) as e:
        print(f"api_tool network error: {e}")
        return "The trivia API is temporarily unavailable. Skip the bonus question and continue the quiz."

    if data.get("response_code") != 0 or not data.get("results"):
        return "Could not fetch a bonus question right now. Skip it and continue the quiz."

    q = data["results"][0]
    question = html.unescape(q["question"])
    answer = html.unescape(q["correct_answer"])

    return f"Bonus question: {question} (correct answer: {answer})"

print("Tool ready:", api_tool.name)


Tool ready: api_tool


In [14]:
# TODO: your third tool, decorated with @function_tool, from your Module 7 assignment
@function_tool
def third_tool(value: str) -> str:
    """Record a numerical quiz score in a file for later review.

    Args:
        value: The numerical quiz score to record.
    """
    try:
        score = int(value)

        with open("score.txt", "a", encoding="utf-8") as f:
            f.write(f"{score}\n")

        print(f"Tool called to record score: {score}")

        return f"Score of {score} was successfully recorded."

    except ValueError:
        return "Please provide a valid numerical score."

print("Tool ready:", third_tool.name)

Tool ready: third_tool


## 5. Build your Agent

**TODO:** Copy this from your Module 7 assignment, unchanged: `name`, `instructions`, your `tools`
list, and `model=azure_model`.

In [15]:
# TODO: instructions = f"""..."""  (your # Your role / # Context / # Rules structured template,
#       using {persona} and {notes}, from your Module 7 assignment)
# TODO: your_agent = Agent(name=..., instructions=instructions, tools=[...], model=azure_model)
instructions = f"""

{persona}
You're rank quiz instructor, you are here to help make this ranking quiz interesting.

use the study notes as the reference for the rank of the video game genres, they are suppose to be displayed randomly and the user suppose
to guess the rank by itself or request a hint to guess it correctly but it can only be used once.

if the student is wrong, type out of the topic or type they don't know, you must record it in my_tool before moving on,
so they can review later.

if the student is asking for a bonus question or another topic you will use api_tool to pull a trivia question from the database.

if the student asks how they are doing you will utilize third_tool to look how many questions they got it right or got it wrong 
and record that in score.txt.

Study notes:
{study_notes}
"""

study_buddy = Agent(
    name="Rank Quiz Instructor",
    instructions=instructions,
    tools=[my_tool, api_tool, third_tool],
    model=azure_model,
)

print("Agent built:", study_buddy.name)

Agent built: Rank Quiz Instructor


## 6. Confirm the problem: two calls that do not know each other

**TODO:** Run your agent twice in a row with `await Runner.run(your_agent, "...")`, the same way
Lab 7, Step 7 did with Study Buddy. Pick a first message that gives the agent some information (for
example, ask it a question whose answer it needs to remember), then a second message that only
makes sense if the agent remembers the first (for example, "what did I just ask you?" or "say that
again"). Confirm the second reply shows the agent has no idea what you mean.

In [30]:
# TODO: result = await Runner.run(your_agent, "...")
# TODO: print(result.final_output)
result = await Runner.run(study_buddy,"For our ranking quiz, my guess for the rank of Horror is 17.")
print(result.final_output)


A bold guess—but not quite.

In our study notes ranking, **Horror is rank 8**, not 17.

Would you like to:
1. guess another rank for **Horror**, or  
2. ask for your **one hint**?


In [31]:
# TODO: result = await Runner.run(your_agent, "...a follow up that only makes sense with memory...")
# TODO: print(result.final_output)
result = await Runner.run(study_buddy, "What rank did I guess for Horror?")
print(result.final_output)

You haven’t given a rank for Horror yet.

If you want, you may now guess its rank, or say “hint” for your one-time hint.


## 7. Fix it with a file backed `SQLiteSession`

**TODO:** Create a `SQLiteSession`, the same way Lab 7, Step 9 did: give it a `session_id` string,
and a `db_path` so the conversation is written to a file and survives a kernel restart. Then repeat
the same two messages from Step 6, this time passing `session=your_session` to both calls, and
confirm the second reply now correctly remembers the first.

In [38]:
# TODO: your_session = SQLiteSession("your_session_id", "your_conversation_history.db")
persistent_session = SQLiteSession("study_buddy_main", "conversation_history.db")


In [39]:
# TODO: result = await Runner.run(your_agent, "...", session=your_session)
# TODO: print(result.final_output)
result = await Runner.run(study_buddy, "For our ranking quiz, my guess for the rank of Horror is 17.", session = persistent_session)
print(result.final_output)

Understood — your remembered guess for **Horror** is **17**.

We are still on this active question:

**What is the rank of: _Shooter_?**


In [40]:
# TODO: result = await Runner.run(your_agent, "...same follow up as Step 6...", session=your_session)
# TODO: print(result.final_output)
result = await Runner.run(study_buddy, "What rank did I guess for Horror?", session = persistent_session)
print(result.final_output)

You guessed **17** for **Horror**.


## 8. Give it a real chat UI with Gradio

**TODO:** Copy this skeleton from Lab 7, Step 11 exactly, structure and all. Notice what is
different from your Module 7 assignment's version: `history` is no longer rebuilt inside `chat()`
at all, `session=your_session` already gives `Runner.run` everything it needs to remember the
conversation.

Try a conversation that would trigger more than one of your three tools across a few turns, then
interrupt the cell (stop button) when you are done chatting.

In [23]:
async def chat(message, history):
    # TODO: result = await Runner.run(your_agent, message, session=your_session)
    # TODO: return result.final_output
    result = await Runner.run(study_buddy, message, session=persistent_session)
    return result.final_output

gr.ChatInterface(chat).launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


Tool called to log a missed question: Missed quiz question: What is the rank of Stealth? Student answered 12.
Tool called to log a missed question: Missed quiz question: What is the rank of RPG? Student said they don't know.
Tool called to log a missed question: Missed quiz question: What is the rank of Adventure? Student said out of topic.


## Reflection Questions

Answer these in a new Markdown cell at the end of your notebook:

1. In Step 6, what exactly did your agent get wrong on the second call, and why? Point to the
   specific piece of information it had no way of knowing.
   -
2. Compare your Gradio `chat()` function here to the one in your Module 7 assignment. What code did
   you get to delete, and what does `session=your_session` do in its place?
   -
3. Your `your_session` was created with a `db_path`. If you had left `db_path` out entirely, what
   would still work the same, and what would break the next time you restarted the kernel?
   -
4. `SQLiteSession` also has a `pop_item()` method, demonstrated in Lab 7, Step 10, for undoing the
   last turn. Describe one realistic situation in your own agent's domain where you would want to
   call it.
   -

## Submission Checklist

- Remove your `.env` file before submitting; it contains your API key.
- Remove any `.db` file your session created; it may contain real conversation content.
- Run the notebook top to bottom and confirm it executes cleanly from a fresh kernel.
- Confirm Step 6 shows the agent failing to remember, and Step 7 shows the same scenario working
  correctly once `session=your_session` is added.
- Answer all four reflection questions in a Markdown cell at the end of the notebook.
- Rename the file to include your name, for example `Module8Assignment_JaneDoe.ipynb`, then submit
  it.